# Khmer History Retrival Argumented Generation

### Import the library 

In [5]:
import sys
!{sys.executable} -m pip install khmer-nltk

In [6]:
import re
import unicodedata
import wikipediaapi
import nltk
from khmernltk import sentence_tokenize



## 1. API Fetching 

In [ ]:
# 1. --- STAGE 1: API FETCHING ---
def fetch_khmer_history_via_api():
    """
    Connects to the official Wikipedia API endpoint, switches to Khmer ('km'),
    and pulls the text data directly while stripping away interface junk.
    """
    # ➡️ REQUIRED: Initialize the API wrapper targeting the Khmer language subdomain ('km')
    # Custom user-agent string avoids connection blocks from Wikipedia servers
    wiki = wikipediaapi.Wikipedia(
        user_agent='KhmerHistoryRAGProject/1.0 (2024384ly@aupp.edu.kh; ITM454 Project)',
        language='km'
    )
    
    # Target title string extracted directly from your Wikipedia link
    target_page_title = "ប្រវត្តិសាស្ត្រកម្ពុជា" 
    
    print(f"📡 Querying Khmer Wikipedia API for page: '{target_page_title}'...")
    page = wiki.page(target_page_title)
    
    if not page.exists():
        raise Exception(f"❌ The page '{target_page_title}' could not be located in the Khmer Wikipedia system.")
        
    print(f"✅ Page Connected! Processing sections for title: {page.title}")
    
    scraped_sections = []
    
    # Wikipedia API naturally groups content by sections, making metadata tracking clean
    # Loop over all primary sub-sections of the historical article
    for section in page.sections:
        section_title = section.title
        section_text = section.text.strip()
        
        # Quality filter: skip empty dividers or short edit templates
        if len(section_text) > 20:
            scraped_sections.append({
                "title": section_title,
                "text": section_text
            })
            
    return scraped_sections, page.fullurl


In [9]:
# 2. --- STAGE 2: LINGUISTIC PREPROCESSING & WINDOW SPLITTING (Owned by Soryapheak LY) ---
def clean_and_normalize_khmer(text: str) -> str:
    """Enforces standard NFC Unicode byte consistency across continuous script blocks."""
    # Normalize character compositions to eliminate silent script mismatch bugs
    normalized_text = unicodedata.normalize('NFC', text)
    # Clear out zero-width spaces (\u200b) and soft hyphens
    normalized_text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', normalized_text)
    # Collapse repetitive line-jumps and whitespace blocks
    normalized_text = re.sub(r'\s+', ' ', normalized_text)
    return normalized_text.strip()

In [10]:
def process_and_split_pipeline(section_data, source_url, chunk_size=750, chunk_overlap=100):
    """
    Groups clean text blocks into sentences with khmer-nltk boundaries and
    applies a sliding context window to map text pieces safely.
    """
    final_knowledge_chunks = []
    
    for section in section_data:
        raw_text = section["text"]
        era_heading = section["title"]
        
        normalized_text = clean_and_normalize_khmer(raw_text)
        
        # Tokenize sentences strictly using native full stops (។)
        sentences = sentence_tokenize(normalized_text)
        
        current_chunk_sentences = []
        current_chunk_length = 0
        
        # Structural metadata payload attached to every vector document node
        metadata = {
            "source_url": source_url,
            "document_domain": "Khmer Wikipedia API",
            "historical_era_heading": era_heading
        }
        
        for sentence in sentences:
            # Approximate Khmer token size using character cluster proxy metrics
            sentence_len_estimate = len(sentence) // 4
            
            if current_chunk_length + sentence_len_estimate > chunk_size:
                if current_chunk_sentences:
                    final_knowledge_chunks.append({
                        "text_payload": " ".join(current_chunk_sentences),
                        "metadata": metadata.copy()
                    })
                # Maintain sliding window memory trace
                overlap_slice = current_chunk_sentences[-max(1, chunk_overlap // 50):]
                current_chunk_sentences = list(overlap_slice)
                current_chunk_length = sum([len(s) // 4 for s in current_chunk_sentences])
                
            current_chunk_sentences.append(sentence)
            current_chunk_length += sentence_len_estimate
            
        # Drain the final leftovers pool loop data block
        if current_chunk_sentences:
            final_knowledge_chunks.append({
                "text_payload": " ".join(current_chunk_sentences),
                "metadata": metadata.copy()
            })
            
    return final_knowledge_chunks

In [11]:

if __name__ == "__main__":
    # Step 1: Pull clean raw segments straight from API weights
    raw_api_sections, live_url = fetch_khmer_history_via_api()
    
    # Step 2: Feed strings directly into the text splitter loop
    processed_rag_chunks = process_and_split_pipeline(raw_api_sections, source_url=live_url)
    
    print("\n" + "="*50)
    print("📊 API PIPELINE RUN DIAGNOSTIC RESULTS")
    print("="*50)
    print(f"Total Database Chunk Nodes Ingested : {len(processed_rag_chunks)}")
    print("-" * 50)
    
    # Verify that metadata context has correctly followed the text chunks
    sample_node = processed_rag_chunks[2] # Inspecting index 2 node payload
    print("📌 [VERIFICATION PROTOTYPE DATA CHECK]:")
    print(f"🏷️ Historical Era Heading Filter : {sample_node['metadata']['historical_era_heading']}")
    print(f"🔗 Reference Citation Link       : {sample_node['metadata']['source_url']}")
    print(f"📄 Text Context Segment Output   : {sample_node['text_payload'][:250]}...")
    print("=" * 50)

📡 Querying Khmer Wikipedia API for page: 'ប្រវត្តិសាស្ត្រកម្ពុជា'...
✅ Page Connected! Processing sections for title: ប្រវត្តិសាស្ត្រកម្ពុជា

📊 API PIPELINE RUN DIAGNOSTIC RESULTS
Total Database Chunk Nodes Ingested : 23
--------------------------------------------------
📌 [VERIFICATION PROTOTYPE DATA CHECK]:
🏷️ Historical Era Heading Filter : **ហ៊្វូណន រឺ ភ្វូណាម
🔗 Reference Citation Link       : https://km.wikipedia.org/wiki/%E1%9E%94%E1%9F%92%E1%9E%9A%E1%9E%9C%E1%9E%8F%E1%9F%92%E1%9E%8F%E1%9E%B7%E1%9E%9F%E1%9E%B6%E1%9E%9F%E1%9F%92%E1%9E%8F%E1%9F%92%E1%9E%9A%E1%9E%80%E1%9E%98%E1%9F%92%E1%9E%96%E1%9E%BB%E1%9E%87%E1%9E%B6
📄 Text Context Segment Output   : ហ៊្វូណន"បានគ្រប់គ្រងផ្លូវគោកយុទ្ធសាស្ត្រក្រៅតែពីតំបន់ឆ្នេរសមុទ្រ" យ៉ាងមានប្រសិទ្ធភាពផងដែរ ហើយក៏បានកាន់កាប់ទីតាំងដ៏លេចធ្លោមួយដែលជា"មជ្ឈមណ្ឌលសេដ្ឋកិច្ច និង រដ្ឋបាល" ស្ថិតនៅចន្លោះបណ្ដាញពាណិជ្ជកម្មតាមមហាសមុទ្រឥណ្ឌា និងចិន ដែលត្រូវបានគេហៅជារួមថា ផ្លូវសូត្...


In [12]:
import json
with open("chunks_preview.json", "w", encoding="utf-8") as f:
    json.dump(processed_rag_chunks, f, ensure_ascii=False, indent=2)

In [13]:
sec = raw_api_sections[2]
clean = clean_and_normalize_khmer(sec["text"])
sents = sentence_tokenize(clean)

print(f"Section: {sec['title']}")
print(f"Sentences: {len(sents)}\n")
for i, s in enumerate(sents[:8]):
    print(f"{i}: ({len(s)} chars) {s[:100]}")

Section: កម្វុជ
Sentences: 10

0: (113 chars) កម្វុជធ្លាប់ ជារដ្ឋចំណុះនៃនគរហ៊្វូណនដែលបានរុលចូលមកដល់ទន្លេមេគង្គចាប់ពីទន្លេមេទឹកភាគខាងជើងឆ្លងកាត់តាម
1: (76 chars) កម្វុជ ក៏ជារដ្ឋឯករាជ្យទីមួយ ដែលបានរីកធំធាត់នៅក្រៅឥទ្ធិពលរបស់អាណាចក្រហ៊្វូណន។
2: (146 chars) កំណត់ត្រាចិនបុរាណបានវែកញែកអំពីស្ដេចពីរអង្គ គឺឝ្រុតវម៌្មនិងឝ្រេស្ឋវម៌្មដែលបានគ្រប់គ្រងនៅឯរាជធានីឝ្រេស
3: (164 chars) ឥទ្ធិពលដ៏ខ្លាំងក្លាបានបង្ហាញឱ្យឃើញនូវអត្តសញ្ញាណរបស់កម្ពុជាត្រូវបានលេចធ្លោឡើង តាមរយៈនគរខ្មែរមួយនៅឯភវប
4: (156 chars) កេរដំណែលនៃនគរនេះគឺបានដោយសារតែស្ដេចមួយអង្គដ៏សំខាន់ជាងគេបង្អស់ នោះគឺ ព្រះបាទឦឝានវម៌្មដែលទ្រង់បានវាយដណ្
5: (77 chars) ទ្រង់បានជ្រើសរើសទីតាំង រាជធានីថ្មីមួយនៅសំបូរព្រៃគុហ៍ដោយបានដាក់ឈ្មោះថាឦឝានបុរ។
6: (155 chars) បន្ទាប់ពីការសោយទិវង្គតនៃព្រះបាទជយវម៌្មទី១ក្នុងឆ្នាំ ៦៨១ មក ភាពចលាចលក៏បានធ្លាក់មកលើនគរនេះ ហើយនិងនៅដើម
7: (84 chars) ព្រះអង្គម្ចាស់បុឝ្ករាក្ឝជាក្សត្រនៅឯសម្ភុបុរបានប្រកាសខ្លួនជាស្ដេចនៃកម្ពុជរដ្ឋទាំងមូល។
